# Recommendation Pipeline Demo

Given a target study, this notebook walks through a recommendation flow used in the project methodology:

1. Find top-K similar studies
2. Identify entities missing from the target
3. Build co-occurrence pairs (target entity -> suggested entity)
4. Apply reranking strategies
5. Print natural-language recommendations from co-occurrence rows (same pattern as the project notebook export in `info.md`)

In [1]:
import sys
from pathlib import Path

root = Path.cwd().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src import recommendations

## 1) Setup target study and candidate pool

Synthetic data is used here.

In [2]:
target_id = "T-100"
target_entities = {"incl_age", "incl_ecog", "ep_os"}

cohort_entities = {
    "T-100": target_entities,
    "N-201": {"incl_age", "incl_ecog", "ep_os", "excl_liver", "ep_pk"},
    "N-202": {"incl_age", "ep_os", "excl_liver", "lab_cbc"},
    "N-203": {"incl_ecog", "ep_os", "excl_liver"},
    "N-204": {"incl_age", "incl_ecog", "ep_os", "excl_cardiac"},
}

similarity_vs_target = {
    "N-201": 0.91,
    "N-202": 0.84,
    "N-203": 0.79,
    "N-204": 0.88,
}

**Cell output(s):**

- Defines target entities and similarity scores for candidate neighbors.

## Top-K neighbors

In [3]:
topk = recommendations.top_k_neighbors(target_id, similarity_vs_target, k=4)
topk

[('N-201', 0.91), ('N-204', 0.88), ('N-202', 0.84), ('N-203', 0.79)]

**Cell output(s):**

- A descending list like `[('N-201', 0.91), ('N-204', 0.88), ...]`.
- These studies form the recommendation neighborhood.

## 2) Missing-entity candidates

These are entities that appear in the top-K neighbors but are absent from the target study.

In [4]:
neighbor_sets = [cohort_entities[sid] for sid, _ in topk]
recommendations.missing_entity_candidates(neighbor_sets, target_entities)

[('excl_liver', 3), ('ep_pk', 1), ('excl_cardiac', 1), ('lab_cbc', 1)]

**Cell output(s):**

- Tuples of `(entity, frequency)` sorted descending.
- Higher frequency means stronger baseline evidence from similar studies.

## 3) Co-occurrence analysis

For each target entity, we count how often suggested entities co-appear in the top-K neighbor set.

Filters:
- `min_neighbor_studies`
- `min_conditional_pct`

In [5]:
cooccurrence_df = recommendations.cooccurrence_pairs(
    target_entities,
    neighbor_sets,
    min_neighbor_studies=1,
    min_conditional_pct=0.0,
)
cooccurrence_df

,entity_in_target,entity_suggested,neighbor_studies_with_pair,conditional_pct
0,ep_os,lab_cbc,1,0.2500
1,ep_os,excl_cardiac,1,0.2500
2,ep_os,ep_pk,1,0.2500
3,ep_os,excl_liver,3,0.7500
4,incl_ecog,excl_cardiac,1,0.3333
5,incl_ecog,ep_pk,1,0.3333
6,incl_ecog,excl_liver,2,0.6667
7,incl_age,lab_cbc,1,0.3333
8,incl_age,excl_cardiac,1,0.3333
9,incl_age,ep_pk,1,0.3333


**Cell output(s):**

- A dataframe with columns like:
  - `entity_in_target`
  - `entity_suggested`
  - `neighbor_studies_with_pair`
  - `conditional_pct`
- This table is the baseline ranking input for rerankers.

## 4) Rerank experiment #1: Specialized deviation

Boost entities that are relatively rare globally but strong in this local neighbor set.

In [6]:
global_counts = {
    "excl_liver": 40,
    "ep_pk": 12,
    "lab_cbc": 8,
    "excl_cardiac": 25,
    "incl_age": 200,
}
recommendations.rerank_specialized_deviation(cooccurrence_df, global_counts)

,entity_in_target,entity_suggested,neighbor_studies_with_pair,conditional_pct,specialized_deviation_score
0,ep_os,lab_cbc,1,0.2500,35.625
1,incl_age,lab_cbc,1,0.3333,35.625
2,ep_os,ep_pk,1,0.2500,23.750
3,incl_age,ep_pk,1,0.3333,23.750
4,incl_ecog,ep_pk,1,0.3333,23.750
5,ep_os,excl_liver,3,0.7500,21.375
6,incl_age,excl_liver,2,0.6667,14.250
7,incl_ecog,excl_liver,2,0.6667,14.250
8,ep_os,excl_cardiac,1,0.2500,11.400
9,incl_ecog,excl_cardiac,1,0.3333,11.400


**Cell output(s):**

- Adds `specialized_deviation_score`.
- Suggested entities are re-ranked by specialized signal instead of raw pair count alone.

## 5) Rerank experiment #2: Network score

Use co-occurrence graph structure (PageRank + betweenness + overlap with target entities).

In [7]:
recommendations.rerank_network_scores(cooccurrence_df, target_entities)

,entity_in_target,entity_suggested,neighbor_studies_with_pair,conditional_pct,pagerank_suggested,betweenness_suggested,target_overlap,network_score
0,incl_age,excl_liver,2,0.6667,0.217284,0.000000,3,1.717284
1,incl_ecog,excl_liver,2,0.6667,0.217284,0.000000,3,1.717284
2,ep_os,excl_liver,3,0.7500,0.217284,0.000000,3,1.717284
3,incl_ecog,ep_pk,1,0.3333,0.105667,0.122222,3,1.636222
4,ep_os,ep_pk,1,0.2500,0.105667,0.122222,3,1.636222
5,ep_os,excl_cardiac,1,0.2500,0.105667,0.122222,3,1.636222
6,incl_ecog,excl_cardiac,1,0.3333,0.105667,0.122222,3,1.636222
7,incl_age,ep_pk,1,0.3333,0.105667,0.122222,3,1.636222
8,incl_age,excl_cardiac,1,0.3333,0.105667,0.122222,3,1.636222
9,ep_os,lab_cbc,1,0.2500,0.077176,0.022222,2,1.082731


**Cell output(s):**

- Adds network-derived columns and `network_score`.
- Produces a graph-aware ranking variant.

## 6) Rerank experiment #3: Specialized network (hub penalty)

Apply an additional penalty to high-degree generic hubs so local-fit entities can move up.

In [8]:
recommendations.rerank_specialized_network(cooccurrence_df, target_entities, hub_penalty=0.5)

,entity_in_target,entity_suggested,neighbor_studies_with_pair,conditional_pct,pagerank_suggested,betweenness_suggested,target_overlap,network_score,hub_adjustment,specialized_network_score
0,ep_os,excl_cardiac,1,0.2500,0.105667,0.122222,3,1.636222,0.785714,1.285603
1,ep_os,ep_pk,1,0.2500,0.105667,0.122222,3,1.636222,0.785714,1.285603
2,incl_ecog,ep_pk,1,0.3333,0.105667,0.122222,3,1.636222,0.785714,1.285603
3,incl_age,ep_pk,1,0.3333,0.105667,0.122222,3,1.636222,0.785714,1.285603
4,incl_age,excl_cardiac,1,0.3333,0.105667,0.122222,3,1.636222,0.785714,1.285603
5,incl_ecog,excl_cardiac,1,0.3333,0.105667,0.122222,3,1.636222,0.785714,1.285603
6,ep_os,lab_cbc,1,0.2500,0.077176,0.022222,2,1.082731,0.857143,0.928055
7,incl_age,lab_cbc,1,0.3333,0.077176,0.022222,2,1.082731,0.857143,0.928055
8,ep_os,excl_liver,3,0.7500,0.217284,0.000000,3,1.717284,0.500000,0.858642
9,incl_age,excl_liver,2,0.6667,0.217284,0.000000,3,1.717284,0.500000,0.858642


## 7) Natural language recommendations

The project export builds one paragraph per co-occurrence row: anchor on an entity the target already has, then name a missing entity that often appears alongside it in similar studies. The template below matches `info.md` (baseline co-occurrence section).

In [9]:
k_neighbors = len(neighbor_sets)
num_nl = 10


def enrich_cooccurrence_for_nl(df, neighbor_entity_sets):
    out = df.copy()
    t_map = {e: sum(1 for s in neighbor_entity_sets if e in s) for e in out["entity_in_target"].unique()}
    s_map = {e: sum(1 for s in neighbor_entity_sets if e in s) for e in out["entity_suggested"].unique()}
    out["target_entity_studies"] = out["entity_in_target"].map(t_map)
    out["recommended_entity_studies"] = out["entity_suggested"].map(s_map)
    out["cooccurrence_pct_display"] = (out["conditional_pct"] * 100).round(1)
    return out


cooc_nl = enrich_cooccurrence_for_nl(cooccurrence_df, neighbor_sets)
cooc_nl = cooc_nl.sort_values("conditional_pct", ascending=False).reset_index(drop=True)

for _, row in cooc_nl.head(num_nl).iterrows():
    t_ent = row["entity_in_target"]
    r_ent = row["entity_suggested"]
    pct = row["cooccurrence_pct_display"]
    n_rec = int(row["recommended_entity_studies"])
    print(
        f"From our data, we see that Studies similar to yours that used `{t_ent}` (which you already have) "
        f"usually paired it up with `{r_ent}` (which you don't have)\n"
        f"- {pct}% of top k similar studies with the entity that you have also include the entity that you don't have. "
        f"The entity that you don't have appears in {n_rec} of {k_neighbors} neighbor studies total\n"
    )

print("---")
print(
    f"Among the top {k_neighbors} similar studies, these entities are missing from `{target_id}` but common in neighbors:"
)
for ent, freq in recommendations.missing_entity_candidates(neighbor_sets, target_entities):
    print(f"- `{ent}` appears in {freq} of {k_neighbors} neighbor studies.")

From our data, we see that Studies similar to yours that used `ep_os` (which you already have) usually paired it up with `excl_liver` (which you don't have)
- 75.0% of top k similar studies with the entity that you have also include the entity that you don't have. The entity that you don't have appears in 3 of 4 neighbor studies total

From our data, we see that Studies similar to yours that used `incl_ecog` (which you already have) usually paired it up with `excl_liver` (which you don't have)
- 66.7% of top k similar studies with the entity that you have also include the entity that you don't have. The entity that you don't have appears in 3 of 4 neighbor studies total

From our data, we see that Studies similar to yours that used `incl_age` (which you already have) usually paired it up with `excl_liver` (which you don't have)
- 66.7% of top k similar studies with the entity that you have also include the entity that you don't have. The entity that you don't have appears in 3 of 4 nei